# 01 — Student Dropout: Data Understanding & EDA

**Mục tiêu:** Khám phá dữ liệu Train bằng Jupyter Notebook, dùng bộ split/fold đã lưu của nhóm.

- Dataset: 4.424 mẫu, 36 đặc trưng, nhãn Dropout / Enrolled / Graduate.
- Split cố định: 80/20, seed 42, stratify; 5-fold CV trong Train.
- **Không dùng Test để chọn đặc trưng, tiền xử lý hoặc mô hình.**
- Notebook này **không huấn luyện mô hình**. Mọi preprocessing có học tham số sẽ được fit riêng trong mỗi training fold sau này.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

# Tìm thư mục final/ dù chạy notebook từ final/notebooks hoặc root repo.
cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents]
project_root = next(
    (p for p in candidates if (p / "src" / "data.py").exists()),
    None,
)
if project_root is None:
    project_root = next(
        (p / "final" for p in candidates if (p / "final" / "src" / "data.py").exists()),
        None,
    )
if project_root is None:
    raise FileNotFoundError("Không tìm thấy final/src/data.py")

sys.path.insert(0, str(project_root))
from src.data import load_student_data, load_splits, read_preprocessing_config

print("Project:", project_root)

## 1. Nạp dữ liệu từ loader chung

Đọc dữ liệu gốc và chỉ số đã lưu. **Không chạy train_test_split lần nữa**; các vị trí CV áp dụng trên Train, không trên toàn bộ CSV.

In [ ]:
X_all, y_all = load_student_data()
train_ids, test_ids, cv_splits, split_manifest = load_splits()

X_train = X_all.iloc[train_ids].reset_index(drop=True)
y_train = y_all.iloc[train_ids].reset_index(drop=True)

print("Train:", X_train.shape)
print("Test size (metadata only):", len(test_ids))
print("CV folds:", len(cv_splits))
assert len(X_train) == split_manifest["n_train"]
assert len(cv_splits) == split_manifest["cv"]["n_splits"]
# Không đọc X_all.iloc[test_ids] hoặc y_all.iloc[test_ids] trong các phân tích dưới đây.

## 2. Xem trước đặc trưng và nhóm kiểu dữ liệu

In [ ]:
display(X_train.head())
print("Số cột theo dtype:")
display(X_train.dtypes.value_counts().rename("count").to_frame())

# Mã số của các biến phân loại vẫn có dtype số, không đồng nghĩa là biến liên tục.
preprocess_info = read_preprocessing_config()
display(pd.DataFrame([
    {"feature_group": group, "n_features": len(preprocess_info[group]["columns"])}
    for group in ["numeric", "categorical", "binary"]
]))

## 3. Phân phối Target trong Train

In [ ]:
class_counts = y_train.value_counts().sort_index()
class_percent = 100 * y_train.value_counts(normalize=True).sort_index()
display(pd.DataFrame({
    "samples": class_counts,
    "percent": class_percent.round(2),
}))

axis = class_counts.plot.bar(title="Class distribution — TRAIN only")
axis.set_xlabel("Target")
axis.set_ylabel("Number of students")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 4. Giá trị thiếu, bản ghi trùng và thống kê

In [ ]:
missing = X_train.isna().sum().sort_values(ascending=False)
display(missing[missing > 0].rename("missing_count").to_frame())

print("Số hàng có cùng giá trị đặc trưng:", int(X_train.duplicated().sum()))
print("Không tự xóa bản ghi khi chưa có lý do nghiệp vụ.")

numeric_cols = preprocess_info["numeric"]["columns"]
display(X_train[numeric_cols].describe().T)

## 5. Kiểm tra phân phối nhãn của 5 validation folds

Mỗi fold chỉ lấy từ Train; nhãn trong Train và Validation không tạo nên bộ chia Test mới.

In [ ]:
fold_summary = []
for fold_number, (_, val_positions) in enumerate(cv_splits, start=1):
    labels = y_train.iloc[val_positions]
    ratios = labels.value_counts(normalize=True)
    fold_summary.append({
        "fold": fold_number,
        "n_validation": len(val_positions),
        **{
            f"pct_{label}": round(100 * ratios.get(label, 0), 2)
            for label in sorted(y_train.unique())
        },
    })
display(pd.DataFrame(fold_summary).set_index("fold"))

## 6. Kết luận EDA và cảnh báo Data Leakage

**Cần thống nhất thời điểm dự đoán:** Nếu dự đoán khi vừa nhập học, không được sử dụng kết quả học kỳ 1 hoặc học kỳ 2 làm đầu vào. Nếu dự đoán sau học kỳ 2, cần định nghĩa rõ thời gian và ý nghĩa của nhãn.

Điền nhận xét sau khi chạy:
- Giá trị thiếu / trùng:
- Mất cân bằng lớp:
- Đặc trưng có nguy cơ leakage:
- Feature engineering dự kiến:
- Quyết định thời điểm dự đoán:
- Câu hỏi cần thống nhất với nhóm:

**Bước tiếp theo:** thiết kế preprocessing chung trong src/features.py; không fit preprocessing trên toàn bộ Train trước khi làm CV.